# AI-Powered Criminal Network Analysis System
## SIH26189 — Ministry of Home Affairs, India

This notebook demonstrates the core data science pipeline behind the CrimeGraph AI platform:
- Dataset generation and exploration
- Graph construction and visualization
- Centrality analysis (Degree, Betweenness, Closeness, PageRank)
- Community detection (Label Propagation)
- Anomaly / Threat scoring
- Key insights and top suspect identification

In [ ]:
import json, os, sys
from collections import defaultdict, Counter
import random

# Add project root to path
sys.path.insert(0, os.path.abspath('..'))
from utils.analyzer import CriminalNetworkAnalyzer, degree_centrality, betweenness_centrality, closeness_centrality, pagerank, community_detection

print('Modules loaded successfully!')

## 1. Load and Explore the Dataset

In [ ]:
# Load all data
analyzer = CriminalNetworkAnalyzer('../data')
analyzer.load_data()

print(f'\n=== Dataset Summary ===')
print(f'  Persons:        {len(analyzer.persons)}')
print(f'  FIRs:           {len(analyzer.firs)}')
print(f'  CDRs:           {len(analyzer.cdrs)}')
print(f'  Transactions:   {len(analyzer.transactions)}')
print(f'  Relationships:  {len(analyzer.relationships)}')
print(f'  Surveillance:   {len(analyzer.surveillance)}')

In [ ]:
# Sample person profile
sample = analyzer.persons[0]
print('=== Sample Person Profile ===')
for k, v in sample.items():
    print(f'  {k:<25} {v}')

In [ ]:
# Crime type distribution
crime_counts = Counter(f['crime_type'] for f in analyzer.firs)
print('=== Crime Type Distribution ===')
for crime, count in crime_counts.most_common(10):
    bar = '|' * count
    print(f'  {crime:<30} {count:>3}  {bar}')

In [ ]:
# Severity distribution
sev_counts = Counter(f['severity'] for f in analyzer.firs)
print('=== FIR Severity Distribution ===')
for sev in ['Critical', 'High', 'Medium', 'Low']:
    count = sev_counts.get(sev, 0)
    pct = count / len(analyzer.firs) * 100
    bar = '#' * int(pct / 2)
    print(f'  {sev:<10} {count:>3}  ({pct:5.1f}%)  {bar}')

## 2. Build the Criminal Network Graph

In [ ]:
# Build the graph
analyzer.build_graph()
g = analyzer.graph

print(f'\n=== Graph Statistics ===')
print(f'  Nodes (suspects):  {len(g.nodes)}')
print(f'  Edges (links):     {len(g.edges)}')

# Degree distribution
degrees = [g.degree(n) for n in g.nodes]
print(f'  Avg degree:        {sum(degrees)/len(degrees):.2f}')
print(f'  Max degree:        {max(degrees)}')
print(f'  Min degree:        {min(degrees)}')

# Edge type distribution
edge_labels = Counter(label for _,_,_,label,_ in g.edges)
print(f'\n=== Edge Types ===')
for label, count in edge_labels.most_common():
    print(f'  {label:<20} {count}')

## 3. Graph Analytics — Centrality Measures

In [ ]:
print('Computing centrality measures (this may take a moment)...')

# Degree centrality
deg = degree_centrality(g)
print('  [OK] Degree centrality computed')

# Betweenness centrality
btw = betweenness_centrality(g)
print('  [OK] Betweenness centrality computed')

# Closeness centrality
clo = closeness_centrality(g)
print('  [OK] Closeness centrality computed')

# PageRank
pgr = pagerank(g)
print('  [OK] PageRank computed')

In [ ]:
# Top 10 by PageRank
top_by_pagerank = sorted(pgr.items(), key=lambda x: -x[1])[:10]
print('=== Top 10 Suspects by PageRank (Influence) ===')
print(f'  {"Rank":<5} {"ID":<6} {"Name":<25} {"PageRank":<12} {"Betweenness":<14} {"Degree"}')
print('  ' + '-'*75)
for rank, (pid, pr) in enumerate(top_by_pagerank, 1):
    person = next((p for p in analyzer.persons if p['id'] == pid), {})
    name = person.get('name', pid)
    print(f'  {rank:<5} {pid:<6} {name:<25} {pr*100:>8.3f}%  {btw.get(pid,0)*100:>10.3f}%  {g.degree(pid)}')

In [ ]:
# Top 10 by Betweenness (Bridge/Broker nodes)
top_by_btw = sorted(btw.items(), key=lambda x: -x[1])[:10]
print('=== Top 10 Broker Suspects (High Betweenness = Info Gatekeepers) ===')
print(f'  {"Rank":<5} {"ID":<6} {"Name":<25} {"Betweenness"}')
print('  ' + '-'*55)
for rank, (pid, bw) in enumerate(top_by_btw, 1):
    person = next((p for p in analyzer.persons if p['id'] == pid), {})
    name = person.get('name', pid)
    print(f'  {rank:<5} {pid:<6} {name:<25} {bw*100:.4f}%')

## 4. Community Detection

In [ ]:
# Run community detection
labels, communities = community_detection(g)

# Filter meaningful communities (size >= 2)
valid_comms = [c for c in communities if len(c) >= 2]
valid_comms.sort(key=lambda c: -len(c))

print(f'=== Community Detection Results ===')
print(f'  Total communities detected: {len(valid_comms)}')
print(f'  Largest community size:     {len(valid_comms[0]) if valid_comms else 0}')

print(f'\n=== Top 5 Communities ===')
for i, comm in enumerate(valid_comms[:5], 1):
    members = [next((p['name'] for p in analyzer.persons if p['id'] == pid), pid) for pid in comm[:5]]
    more = f" +{len(comm)-5} more" if len(comm) > 5 else ""
    print(f'  Community {i}: {len(comm)} members — {', '.join(members)}{more}')

## 5. Threat Scoring — AI Composite Analysis

In [ ]:
# Compute all metrics
analyzer.compute_metrics()

# Top 10 suspects by composite threat score
top_suspects = analyzer.get_top_suspects(10)

print('=== TOP 10 HIGH-RISK SUSPECTS (AI Threat Score) ===')
print(f'  {"Rank":<5} {"ID":<6} {"Name":<25} {"Score":<10} {"Role":<15} {"Status":<12} {"FIRs"}')
print('  ' + '-'*85)
for i, p in enumerate(top_suspects, 1):
    score_bar = '[' + '#' * int(p['computed_threat_score'] * 20) + '.' * (20 - int(p['computed_threat_score'] * 20)) + ']'
    print(f'  {i:<5} {p["id"]:<6} {p["name"]:<25} {p["computed_threat_score"]*100:>5.1f}%   {p.get("role","?"):<15} {p.get("status","?"):<12} {p["fir_count"]}')

In [ ]:
# Threat score distribution
scores = [p['computed_threat_score'] for p in analyzer.persons]
bins = {'0.0-0.2': 0, '0.2-0.4': 0, '0.4-0.6': 0, '0.6-0.8': 0, '0.8-1.0': 0}
for s in scores:
    if s < 0.2: bins['0.0-0.2'] += 1
    elif s < 0.4: bins['0.2-0.4'] += 1
    elif s < 0.6: bins['0.4-0.6'] += 1
    elif s < 0.8: bins['0.6-0.8'] += 1
    else: bins['0.8-1.0'] += 1

print('=== Threat Score Distribution ===')
for rng, count in bins.items():
    bar = '|' * (count * 3)
    print(f'  {rng}  {count:>3}  {bar}')

## 6. Financial Intelligence Analysis

In [ ]:
# Suspicious financial transactions
suspicious_txns = [t for t in analyzer.transactions if t.get('suspicious')]
hawala_txns = [t for t in analyzer.transactions if t.get('channel') == 'Hawala']
crypto_txns = [t for t in analyzer.transactions if t.get('channel') == 'Cryptocurrency']

total_amount = sum(t.get('amount', 0) for t in analyzer.transactions)
suspicious_amount = sum(t.get('amount', 0) for t in suspicious_txns)

print('=== Financial Intelligence Summary ===')
print(f'  Total transactions:         {len(analyzer.transactions)}')
print(f'  Suspicious transactions:    {len(suspicious_txns)} ({len(suspicious_txns)/len(analyzer.transactions)*100:.1f}%)')
print(f'  Hawala transactions:        {len(hawala_txns)}')
print(f'  Cryptocurrency:             {len(crypto_txns)}')
print(f'  Total amount tracked:       INR {total_amount:,.0f}')
print(f'  Suspicious amount:          INR {suspicious_amount:,.0f}')

# Top 5 suspicious transactions
top_txns = sorted(suspicious_txns, key=lambda t: -t.get('amount', 0))[:5]
print(f'\n=== Top 5 Suspicious Transactions ===')
for t in top_txns:
    print(f'  {t["txn_id"]}  {t["sender_name"]:<20} -> {t["receiver_name"]:<20}  {t["currency"]} {t["amount"]:>15,.0f}  ({t["channel"]})')

## 7. CDR Communication Pattern Analysis

In [ ]:
# Most frequent callers
caller_counts = Counter(c['caller_id'] for c in analyzer.cdrs)
susp_caller_counts = Counter(c['caller_id'] for c in analyzer.cdrs if c.get('suspicious'))

print('=== Top 10 Most Active Communicators ===')
for pid, count in caller_counts.most_common(10):
    person = next((p for p in analyzer.persons if p['id'] == pid), {})
    susp = susp_caller_counts.get(pid, 0)
    print(f'  {pid}  {person.get("name","?"):<25}  Total: {count:>3}  Suspicious: {susp:>3}')

print(f'\n=== CDR Statistics ===')
print(f'  Total CDRs:             {len(analyzer.cdrs)}')
print(f'  Suspicious calls:       {sum(1 for c in analyzer.cdrs if c.get("suspicious"))}')
print(f'  Call types: {dict(Counter(c["call_type"] for c in analyzer.cdrs))}')

## 8. Full Network Statistics

In [ ]:
stats = analyzer.get_network_stats()
print('=== COMPLETE NETWORK STATISTICS ===')
for k, v in stats.items():
    print(f'  {k:<30} {v}')

print('\n[System ready] Launch app.py to view the full interactive dashboard.')